In [0]:
# ── Imports y configuración ──────────────────────────────────────────────
from pyspark.sql import functions as F
from lib.config import Config

cfg = Config()
LANDING = f"{cfg.landing_path}/transactions"

# ── Parámetro: qué lote soltar ───────────────────────────────────────────
# Si existía el desplegable viejo (sin la opción "todos"), lo eliminamos
try:
    dbutils.widgets.remove("lote")
except Exception:
    pass

# "todos" suelta los 5 lotes seguidos
dbutils.widgets.dropdown("lote", "todos", ["0", "1", "2", "3", "4", "todos"], "Lote a soltar")
dbutils.widgets.dropdown("master_snapshot", "1", ["1", "2"], "Master data snapshot")

In [0]:
# ── Datos del "sistema origen" ───────────────────────────────────────────
# Simulamos las columnas de auditoría de un ERP real:
#   createdAt: cuándo se creó la venta
#   updatedAt: cuándo se modificó por última vez (NULL si nunca se modificó)
src = (spark.table("samples.bakehouse.sales_transactions")
       .withColumn("createdAt", F.col("dateTime"))
       .withColumn("updatedAt", F.lit(None).cast("timestamp")))

# Reparte las ventas en 5 lotes estables
lotes = src.withColumn("lote", F.abs(F.hash("transactionID")) % 5)


# ── Utilidad: ¿existe la carpeta? (para no regenerar un lote ya entregado) ──
def existe(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


# ── Genera los archivos JSON de un lote (simula la extracción de un día) ──
def soltar_lote(n: int):
    destino = f"{LANDING}/lote_{n}"
    if existe(destino):
        print(f"Lote {n} ya existe: no se vuelve a generar (idempotencia)")
        return

    df = lotes.filter(F.col("lote") == n).drop("lote")

    # Lote 2: el origen agrega una columna nueva sin avisar (schema drift)
    if n == 2:
        df = df.withColumn("discountPct", F.round(F.rand(seed=42) * 0.15, 2))

    # Lote 3: 20 ventas del lote 0 MODIFICADAS en el origen
    if n == 3:
        corr = (lotes.filter("lote = 0").limit(20).drop("lote")
                .withColumn("quantity", F.col("quantity") + 1)
                .withColumn("totalPrice", F.col("quantity") * F.col("unitPrice"))
                .withColumn("updatedAt", F.current_timestamp()))
        df = df.unionByName(corr)

    # Lote 4: filas inválidas (cantidad negativa y cliente nulo)
    if n == 4:
        bad_qty = (df.limit(3)
                   .withColumn("transactionID", F.col("transactionID") + 10_000_000)
                   .withColumn("quantity", F.lit(-1)))
        bad_cust = (df.limit(2)
                    .withColumn("transactionID", F.col("transactionID") + 20_000_000)
                    .withColumn("customerID", F.lit(None).cast("long")))
        df = df.unionByName(bad_qty).unionByName(bad_cust)

    # Escritura JSON (ignoreNullFields=false: escribe "updatedAt": null en vez de omitirlo)
    df.write.option("ignoreNullFields", "false").json(destino)
    print(f"Lote {n}: {df.count()} filas escritas en {destino}")

    # Lote 4: archivo extra con un tipo incorrecto (quantity como texto)
    if n == 4:
        dbutils.fs.put(
            f"{LANDING}/lote_4_tipo_erroneo/part-00000.json",
            '{"transactionID": 30000001, "customerID": 1000000, "franchiseID": 3000000, '
            '"dateTime": "2024-05-20T10:00:00", "product": "Croissant", '
            '"quantity": "dos", "unitPrice": 3, "totalPrice": 6, "paymentMethod": "visa", '
            '"createdAt": "2024-05-20T10:00:00", "updatedAt": null}\n',
            overwrite=True)
        print("Lote 4: + 1 archivo con quantity como texto")

In [0]:
# ── Ejecutar: un lote concreto o todos seguidos ──────────────────────────
lote = dbutils.widgets.get("lote")
for n in (range(5) if lote == "todos" else [int(lote)]):
    soltar_lote(n)

In [0]:
# ── Verificación: qué carpetas de lotes hay en landing ───────────────────
display(dbutils.fs.ls(LANDING))

In [0]:
# =============================================================================
# Master data extracts (full load): customers and franchises
# -----------------------------------------------------------------------------
# Each extract is a complete snapshot of the source table, written to its own
# folder (snapshot_v<N>). Snapshot 2 simulates changes in the source system:
#   - 1 in 50 customers moves to Barcelona  -> SCD2-tracked attributes
#   - 1 in 40 customers changes email       -> SCD1 attribute (no history)
# =============================================================================

def drop_master_snapshot(version: int) -> None:
    """Write a full snapshot of customers and franchises to landing (idempotent)."""
    customers = spark.table("samples.bakehouse.sales_customers")
    franchises = spark.table("samples.bakehouse.sales_franchises")

    if version >= 2:
        moves = F.col("customerID") % 50 == 0
        customers = (customers
            .withColumn("city",    F.when(moves, F.lit("Barcelona")).otherwise(F.col("city")))
            .withColumn("state",   F.when(moves, F.lit("Catalonia")).otherwise(F.col("state")))
            .withColumn("country", F.when(moves, F.lit("Spain")).otherwise(F.col("country")))
            .withColumn("email_address",
                        F.when(F.col("customerID") % 40 == 0,
                               F.concat(F.lit("new."), F.col("email_address")))
                         .otherwise(F.col("email_address"))))

    for name, df in {"customers": customers, "franchises": franchises}.items():
        target = f"{cfg.landing_path}/{name}/snapshot_v{version}"
        if existe(target):
            print(f"{name} snapshot v{version} already exists: skipped (idempotent)")
            continue
        df.write.option("ignoreNullFields", "false").json(target)
        print(f"{name} snapshot v{version}: {df.count()} rows -> {target}")


drop_master_snapshot(int(dbutils.widgets.get("master_snapshot")))